# Network metric calculation

This notebook calculates network measures for the semantic networks constructed from cosine similarity matrices. It reads the saved GraphML files, computes structural metrics for each language and network construction method, and saves the results in a summary CSV file. The calculated measures include graph size, degree structure, connectivity, distance measures on the largest connected component, clustering, community structure using the Leiden algorithm, and degree assortativity.

In [ ]:
import os
import time
import logging

import pandas as pd
import igraph as ig
import leidenalg as la
import numpy as np

ROOT = "data"

graph_dir = os.path.join(ROOT, "igraph_matrices")
output_dir = os.path.join(ROOT, "metrics_outputs")
os.makedirs(output_dir, exist_ok=True)

log_file = os.path.join(output_dir, "metrics_processing.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

In [ ]:

metrics_csv= os.path.join(output_dir, "igraph_metrics_summary.csv")

if os.path.exists(metrics_csv):
    df_existing = pd.read_csv(metrics_csv)
    processed_langs = set(df_existing["language"].tolist())
    metrics_summary = df_existing.to_dict(orient="records") 
    logging.info(f"Loaded existing summary with {len(processed_langs)} languages")
else:
    df_existing = pd.DataFrame()
    processed_langs = set()
    metrics_summary = []
    logging.info("No previous summary file found.")


for file in os.listdir(graph_dir): 

    lang_code = file.split(".")[0]
    method = file.split(".")[-2]
    start_time = time.time()

    logging.info(f" Processing language: {lang_code}")

    file_path =  os.path.join(graph_dir, file)

    try:
        # Load the saved semantic network as an igraph GraphML file.
        G_ig = ig.Graph.Read_GraphML(file_path)

        logging.info(f" Converted NetworkX to igraph for {lang_code} | "
                     f"Nodes: {G_ig.vcount()}, Edges: {G_ig.ecount()}")

         #1. Basic Network Metrics
        num_nodes = G_ig.vcount()
        num_edges = G_ig.ecount()
        density = G_ig.density()

        degrees = G_ig.degree()
        avg_degree = float(np.mean(degrees))
        median_degree = float(np.median(degrees))
        degree_std = float(np.std(degrees))
        max_degree = int(np.max(degrees))

        
        # 2.Connectivity and  distance metrics on the giant component because both diameter and average path length require connected graphs.
        components = G_ig.components() 
        num_components = len(components) 
        giant = components.giant() 
        largest_component_nodes = giant.vcount() 
        largest_component_ratio = largest_component_nodes / num_nodes if num_nodes > 0 else 0 

        if giant.vcount() > 1: 
            diameter_gc = giant.diameter() 
            avg_shortest_path_gc = giant.average_path_length() 
        else:
            diameter_gc = None 
            avg_shortest_path_gc = None 
        

        # 3. Clustering metrics
        avg_local_clustering = G_ig.transitivity_avglocal_undirected()
        global_transitivity = G_ig.transitivity_undirected()


        #4. Community detection (Leiden algorithm) 
        try:
           partition = la.find_partition(G_ig, la.ModularityVertexPartition)
           modularity = partition.modularity
           num_communities = len(partition)
        except Exception as e:
           logging.error(f" Leiden failed for {lang_code}: {e}")
           modularity = None
           num_communities = None

        # 5. Degree assortativity
        try:
           assortativity = G_ig.assortativity_degree()
        except Exception as e:
           logging.warning(f"Assortativity failed for {lang_code}: {e}")
           assortativity = None


        # Save metrics
        metrics_summary.append({
            "language": lang_code,
            "method": method,
            "nodes": num_nodes,
            "edges": num_edges,
            "density": density,
            "avg_degree": avg_degree,
            "median_degree": median_degree,
            "degree_std": degree_std,
            "max_degree": max_degree,
            "num_components": num_components,
            "largest_component_nodes": largest_component_nodes,
            "largest_component_ratio": largest_component_ratio,
            "diameter": diameter_gc,
            "avg_shortest_path": avg_shortest_path_gc,
            "avg_clustering_coefficient": avg_local_clustering,
            "transitivity": global_transitivity,
            "modularity": modularity,
            "num_communities": num_communities,
            "degree_assortativity": assortativity,
        })

        pd.DataFrame(metrics_summary).to_csv(metrics_csv, index=False)
        
        elapsed_time = time.time() - start_time
        logging.info(f"Finished processing {lang_code} in {elapsed_time:.2f}s")

    except Exception as e:
        logging.error(f" Error in {lang_code}: {e}")

df_final = pd.DataFrame(metrics_summary)
df_final.to_csv(metrics_csv, index=False)
logging.info(f"metrics summary saved: {metrics_csv}")